# Does the multi-stage new LTF scheduler reproduce the LTF spectrum of a GRACE Follow-On ranging record?

`new_ltf_plan` changes how resolution and averaging are distributed across the band without changing what is estimated. Its docstring describes three regimes in one loop: the LTF compromise at low frequency; a middle regime in which the segment length decreases exponentially toward `Lmin`, aiming at about `Jdes` points in total; and a high-frequency regime in which the segment length is clamped at `Lmin` and the frequencies form a linear ramp. Both schedulers estimate the same one-sided ASD, so on the same data the two spectra should agree in level and differ only in bin spacing and scatter. The comparison can fail at a regime boundary, where a stitching error would appear as a step, or where segments become short enough that the window's main lobe sets the resolution.

The record is GRACE Follow-On laser ranging interferometer data for calendar week 52 of 2022 (the `RawPhase` column of `data.mat`), sampled at $f_s = 9.664$ Hz.

`compute_spectrum(x, fs=fs, win="kaiser", olap=0.75, Jdes=1000, force_target_nf=True, scheduler=...)` runs twice: with `scheduler="vectorized_ltf"`, the analyzer default and the reference labeled LTF, and with `scheduler="new_ltf"`. The Kaiser window uses the default 200 dB peak sidelobe level, the default detrend `order=0` removes each segment's mean, and `Lmin` and `bmin` keep their defaults of 1. With `force_target_nf`, each analyzer searches for the `Jdes` that gives 1000 frequencies, so both spectra have the same number of bins. `SpectrumResult.plot` draws the one-sided ASD.

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

from speckit import compute_spectrum
from speckit.plotting import default_rc

plt.rcParams.update(default_rc)

In [ ]:
# The following link should lead to a .mat datafile
# Contact the author at mikedovale@pm.me if the link is broken
url = "https://www.dropbox.com/s/znmk8tfm0gv8a7b/GFO_Week52_2022.mat?dl=1"

# Download the file
response = requests.get(url)
with open("data.mat", "wb") as f:
    f.write(response.content)

# Load the .mat file into a NumPy array
data = loadmat("data.mat")
x = np.array(data["RawPhase"][:, 0])
print(x)

fs = 9.664

In [ ]:
xlabel = "Sample"
ylabel = "Signal"
title = "GRACE Follow On LRI range in meters - Calendar week 52, 2022"

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(x, label="test", color="k")
ax.set_xlabel(xlabel)
ax.set_ylabel(ylabel)
ax.set_title(title)
fig.tight_layout()
fig.align_ylabels()
plt.show()

In [ ]:
olap = 0.75
Jdes = 1000
force_target_nf = True
res = {}

res["ltf"] = compute_spectrum(
    x,
    fs=fs,
    win="kaiser",
    olap=olap,
    Jdes=Jdes,
    force_target_nf=force_target_nf,
    scheduler="vectorized_ltf",
)

In [ ]:
res["new_ltf"] = compute_spectrum(
    x,
    fs=fs,
    win="kaiser",
    olap=olap,
    Jdes=Jdes,
    force_target_nf=force_target_nf,
    scheduler="new_ltf",
)

In [ ]:
fig, ax = plt.subplots()

res["ltf"].plot(ax=ax, label="LTF")
res["new_ltf"].plot(ax=ax, label="New LTF", ls='--')

ax.legend()
plt.show()

## Reading the result

The data cell prints the loaded array, and the next figure shows the time series. The two `compute_spectrum` cells produce no output. The final figure overlays the ASD from the LTF plan (solid) and from the new LTF plan (dashed).

The comparison that decides the notebook is the level of the two traces across the band. At low frequency both schedulers use the same compromise logic, so the traces should nearly coincide. Past the transition out of that regime, `new_ltf_plan` sets the segment length by its own exponential schedule, so the traces can differ in bin spacing and scatter, but not in level. A step at a regime boundary, or an offset over a band, would point at the scheduler rather than the data.

Limits in the code: with `Lmin` at its default of 1, the middle regime shortens segments toward very few samples, where the Kaiser main lobe is wide and the resolution coarse. The download cell fetches the file from Dropbox on every run and overwrites `data.mat` without checking it; the windows notebook reads the same file name.